# PEE — GLiNER2.5 su brevi passaggi di recensioni reali

Questo notebook si esegue in **Google Colab**, anche da iPhone. Scarica autonomamente i pesi del modello; non occorre trasferire `model.safetensors` nella chat.

I tre passaggi sono citazioni **brevi e pubblicate** da RTINGS (Maytag, LG, COSORI), selezionate a mano. **Non** sono recensioni complete, né annotazioni indipendenti, né un benchmark della qualità di PEE. Il risultato JSON non va usato per modificare il database scientifico o il ranking.

**Da iPhone:** collega Colab, poi dal menu **Runtime / Esecuzione** scegli **Run all / Esegui tutto**. Alla fine conferma il download del piccolo JSON e invialo nella conversazione. Non modificare alcuna cella.


## 1. Installazione del motore nel runtime Colab

In [ ]:
%pip -q install "gliner2[local]"

## 2. Download automatico e caricamento del modello

In [ ]:
import importlib.metadata
from gliner2 import AutoExtractor

MODEL_ID = "fastino/gliner2.5-small-v1"
model = AutoExtractor.from_pretrained(MODEL_ID, map_location="cpu")
print("Loader:", type(model).__name__)
print("Architettura:", model.config.architecture)
print("Versione gliner2:", importlib.metadata.version("gliner2"))
assert model.config.architecture == "boundary"

## 3. Tre brevi estratti editoriali autentici

I testi e i collegamenti sono inseriti; non devi copiare nulla. Le pagine di RTINGS possono cambiare nel tempo. La fonte, la data e il testo dovranno essere riesaminati prima di una valutazione formale.


In [ ]:
samples = {
    "maytag": {
        "source_url": "https://www.rtings.com/refrigerator/reviews/maytag/mrt311fffh-black",
        "source_title": "Maytag MRT311FFFE Refrigerator Review",
        "input": "The unit we bought and tested is an MRT311FFFE, with a black finish.",
    },
    "lg": {
        "source_url": "https://www.rtings.com/refrigerator/reviews/lg/studio-srfb27w3",
        "source_title": "LG STUDIO SRFB27W3 Review",
        "input": "The LG STUDIO SRFB27W3 provides great refrigerator capacity.",
    },
    "cosori": {
        "source_url": "https://www.rtings.com/air-fryer/reviews/cosori/dual-blaze-6-8qt-caf-p583s-single-basket",
        "source_title": "COSORI Dual Blaze Review",
        "input": "The COSORI Dual Blaze Air Fryer preheats quickly and offers an excellent cooking speed.",
    },
}
labels = {
    "product_model": "A specific commercially sold product model or model identifier, including its brand",
    "brand": "Manufacturer or consumer product brand name",
    "product_feature": "Physical product component or operational feature",
    "measured_value": "Numerical test result with its measurement unit",
}
print("Publisher excerpts:", len(samples), "— lengths:", {k:len(v["input"]) for k,v in samples.items()})


## 4. Estrazione neurale e controllo degli offset

In [ ]:
import json, time
outputs = {}
for case_id, item in samples.items():
    review_text = item["input"]
    start_time = time.perf_counter()
    extraction = model.extract_entities(review_text, labels, include_confidence=True, include_spans=True)
    duration = round(time.perf_counter() - start_time, 3)
    entities = {key: extraction.get("entities", {}).get(key, []) for key in labels}
    offset_errors = []
    for entity_type, mentions in entities.items():
        for mention in mentions:
            a,b = mention.get("start"), mention.get("end")
            if not (isinstance(a,int) and isinstance(b,int) and 0<=a<b<=len(review_text) and review_text[a:b]==mention.get("text")):
                offset_errors.append({"type":entity_type,"mention":mention})
    outputs[case_id] = {"source_url":item["source_url"],"source_title":item["source_title"],
         "input":review_text,"entities":entities,"seconds_inference":duration,"offset_errors":offset_errors}
    print(case_id, "seconds:", duration, "spans:", {k:len(v) for k,v in entities.items()}, "offset errors:", len(offset_errors))


## 5. Salva un JSON piccolo, senza i pesi del modello

In [ ]:
from pathlib import Path
import platform, importlib.metadata
result = {
    "experiment": "PEE GLiNER2.5 Small authentic short-excerpt pilot",
    "model_id": MODEL_ID,
    "model_architecture": model.config.architecture,
    "gliner2_version": importlib.metadata.version("gliner2"),
    "python_version": platform.python_version(),
    "case_status": "published_short_excerpts_not_independent_benchmark",
    "model_integration_status": "standalone_colab_not_connected_to_PEE",
    "cases": outputs,
}
path = Path('/content/PEE-GLiNER25-real-excerpts-results.json')
path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print("Output:",path.name, "bytes:",path.stat().st_size)
from google.colab import files
files.download(str(path))
